# TP Jour 3 — CNN et classification d'images : cats vs dogs

Trois itérations sur le même problème de classification binaire :

1. **TP1** : CNN from scratch — on veut *voir* l'overfitting
2. **TP2** : data augmentation + Dropout — on le réduit
3. **TP3** : transfer learning MobileNetV2 — on le dépasse

**Adaptation locale** : le TP prévoit Colab + API Kaggle. Je travaille ici en local
(CPU), donc je télécharge le zip **officiel Microsoft** du dataset Cats vs Dogs
(787 Mo, 12 500 images par classe, pas de compte Kaggle requis), je filtre les
JPEG corrompus (défaut connu de ce dataset) et je sous-échantillonne à
**1 500 images par classe** pour un entraînement CPU raisonnable — le critère
« au moins 1 000 images par classe » du sujet reste respecté. `IMG_SIZE=(96, 96)`
pour les TP1/TP2 (l'énoncé autorise 64 à 160).
Pour TensorBoard : `tensorboard --logdir logs` puis http://localhost:6006 (les
magics `%tensorboard` sont propres à Colab/Jupyter interactif).

## Phase 1.1 — Setup + organisation du dataset

In [ ]:
# === CONFIGURATION DU PROJET ===
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "data"   # racine locale (pas /content : on n'est pas sur Colab)

In [ ]:
import os, shutil, random, zipfile
import tensorflow as tf

# Zip officiel Microsoft du dataset Cats vs Dogs (~787 Mo) :
# telechargement direct, pas de kaggle.json necessaire.
zip_url = ("https://download.microsoft.com/download/3/E/1/"
           "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
zip_path = tf.keras.utils.get_file("kagglecatsanddogs_5340.zip", origin=zip_url)

raw_dir = "raw_dataset"
if not os.path.isdir(os.path.join(raw_dir, "PetImages")):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(raw_dir)
src = os.path.join(raw_dir, "PetImages")

# TODO 1 : lister les fichiers bruts, en filtrant les JPEG corrompus.
# Defaut connu de ce dataset : ~1500 fichiers illisibles (zero octet,
# en-tete invalide). On ne garde que les fichiers avec un en-tete JFIF,
# sinon image_dataset_from_directory planterait en plein entrainement.
def list_valid(folder):
    valid = []
    for f in os.listdir(folder):
        path = os.path.join(folder, f)
        try:
            with open(path, "rb") as fh:
                if b"JFIF" in fh.read(10):
                    valid.append(path)
        except OSError:
            pass
    return valid

files_a = list_valid(os.path.join(src, "Cat"))
files_b = list_valid(os.path.join(src, "Dog"))
print(f"Images valides : {len(files_a)} {CLASS_A}, {len(files_b)} {CLASS_B}")

# TODO 3 (avance) : melanger avec seed=42 (reproductibilite), puis
# sous-echantillonner a 1500 images/classe : machine CPU-only, et le critere
# 'au moins 1000 images par classe' du sujet reste respecte.
random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)
files_a = files_a[:1500]
files_b = files_b[:1500]

# TODO 2 : creer les dossiers train/val par classe
for split in ("train", "val"):
    for cls in (CLASS_A, CLASS_B):
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

# TODO 4 : split 80/20 et copie
def dispatch(files, cls):
    n_train = int(len(files) * 0.8)
    for i, f in enumerate(files):
        split = "train" if i < n_train else "val"
        dst = os.path.join(DATA_ROOT, split, cls, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst)

dispatch(files_a, CLASS_A)
dispatch(files_b, CLASS_B)

# Verification : ce bloc doit tourner sans modification
for split in ['train', 'val']:
    for cls in [CLASS_A, CLASS_B]:
        path = os.path.join(DATA_ROOT, split, cls)
        print(f"{path} : {len(os.listdir(path))} images")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 3 images de chaque classe cote a cote, verification visuelle + shapes RGB
plt.figure(figsize=(10, 6))
for row, cls in enumerate([CLASS_A, CLASS_B]):
    folder = os.path.join(DATA_ROOT, "train", cls)
    for col, fname in enumerate(sorted(os.listdir(folder))[:3]):
        img = mpimg.imread(os.path.join(folder, fname))
        plt.subplot(2, 3, row * 3 + col + 1)
        plt.imshow(img)
        plt.title(f"{cls} {img.shape}")
        plt.axis('off')
plt.tight_layout()
plt.savefig("sample_grid.png", dpi=100)
plt.show()

## Phase 1.2 — Preprocessing : normalisation + batching

In [ ]:
IMG_SIZE = (96, 96)   # compromis qualite/temps CPU (sujet : 64 a 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=True, seed=42,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=False, seed=42,
)

# Normalisation [0, 255] -> [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# cache() : evite de relire le disque a chaque epoch (3-5x plus lent sinon)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

images, labels = next(iter(train_ds))
print(f"Shape images batch : {images.shape}")
print(f"Shape labels batch : {labels.shape}")
print(f"Valeurs : min={images.numpy().min():.2f}, max={images.numpy().max():.2f}")

## Phase 1.3 — Architecture CNN from scratch

Calculs faits **avant** d'exécuter (padding `same`, pooling 2x2) :
- bloc 1 : 96x96x3 → conv 32 → pool → **48x48x32**
- bloc 2 : → conv 64 → pool → **24x24x64**
- bloc 3 : → conv 128 → pool → **12x12x128**
- Flatten : 12 × 12 × 128 = **18 432** éléments
- Dense(128) : 18 432 × 128 + 128 = **2 359 424 paramètres** (l'écrasante
  majorité des paramètres du modèle est dans cette couche)

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_scratch(input_shape):
    """CNN from scratch pour la classification binaire.
    Architecture deliberement simple : on veut voir l'overfitting, pas l'eviter.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        # bloc 1 -> (48, 48, 32)
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # bloc 2 -> (24, 24, 64)
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        # bloc 3 -> (12, 12, 128)
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),                       # 12*12*128 = 18432
        layers.Dense(128, activation='relu'),   # 18432*128+128 = 2 359 424
        layers.Dense(1, activation='sigmoid'),  # sortie binaire — sans elle,
        # BCE recevrait des logits non bornes et l'entrainement plafonnerait a 50%
    ])
    return model

model_scratch = build_cnn_scratch(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_scratch.summary()
model_scratch.compile(optimizer='adam', loss='binary_crossentropy',
                      metrics=['accuracy'])

## Phase 1.4 — Entraînement et diagnostic overfitting

In [ ]:
import datetime, time

log_dir = "logs/scratch/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir,
                                                      histogram_freq=1)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True)

start = time.time()
history_scratch = model_scratch.fit(
    train_ds, epochs=20, validation_data=val_ds,
    callbacks=[tensorboard_callback, early_stopping], verbose=2)
training_time_scratch = time.time() - start
print(f"Temps d'entrainement : {training_time_scratch:.0f}s")
print(f"val_accuracy finale : {max(history_scratch.history['val_accuracy']):.3f}")

In [ ]:
def plot_history(history, title=""):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['loss'], label='Train loss')
    ax1.plot(history.history['val_loss'], label='Val loss')
    ax1.set_title(f'{title} - Loss')
    ax1.legend()
    ax2.plot(history.history['accuracy'], label='Train accuracy')
    ax2.plot(history.history['val_accuracy'], label='Val accuracy')
    ax2.set_title(f'{title} - Accuracy')
    ax2.legend()
    plt.tight_layout()
    plt.savefig(f"curves_{title.lower().replace(' ', '_')}.png", dpi=100)
    plt.show()

plot_history(history_scratch, "CNN scratch")

**Lecture des courbes** : la train_accuracy continue de monter pendant que la
val_accuracy plafonne et que la val_loss remonte — c'est l'overfitting classique
d'un CNN sans régularisation sur ~2 400 images. Le point où la val_loss commence
à remonter est le point optimal que l'EarlyStopping attrape
(`restore_best_weights=True` récupère ces poids-là).

## Phase 2.1 — Pipeline d'augmentation

In [ ]:
# Ces couches sont actives uniquement en training (model.fit) et passives en
# inference (predict/evaluate) : la val_accuracy est calculee sur les images
# ORIGINALES, c'est voulu.
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),   # un chat retourne reste un chat
    layers.RandomRotation(0.1),        # ~36 deg max : realiste pour des photos
    layers.RandomZoom(0.1),
], name="data_augmentation")
# Pas de RandomFlip("vertical") : photos naturelles orientees tete en haut,
# un chien a l'envers n'existe pas dans le val set.

sample_image = next(iter(train_ds))[0][0]
plt.figure(figsize=(8, 8))
for i in range(9):
    augmented = data_augmentation(tf.expand_dims(sample_image, 0), training=True)
    plt.subplot(3, 3, i + 1)
    plt.imshow(augmented[0])
    plt.axis('off')
plt.tight_layout()
plt.savefig("augmentation_grid.png", dpi=100)
plt.show()

## Phase 2.2 — Intégration Dropout + réentraînement

In [ ]:
def build_cnn_augmented(input_shape):
    """Meme architecture que build_cnn_scratch + augmentation + Dropout."""
    model = models.Sequential([
        layers.Input(shape=input_shape),
        data_augmentation,
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        # Dropout APRES Flatten : couper des feature maps spatiales entieres
        # (avant Flatten) detruirait des positions ; ici on ne coupe que des
        # activations aplaties. 0.4 = bon range CNN (0.8 empecherait d'apprendre).
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    return model

model_augmented = build_cnn_augmented(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model_augmented.compile(optimizer='adam', loss='binary_crossentropy',
                        metrics=['accuracy'])

log_dir_aug = "logs/augmented/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback_aug = tf.keras.callbacks.TensorBoard(log_dir=log_dir_aug,
                                                          histogram_freq=1)
early_stopping_aug = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True)

start = time.time()
history_augmented = model_augmented.fit(
    train_ds, epochs=20, validation_data=val_ds,
    callbacks=[tensorboard_callback_aug, early_stopping_aug], verbose=2)
training_time_augmented = time.time() - start
print(f"Temps d'entrainement : {training_time_augmented:.0f}s")
print(f"val_accuracy finale : {max(history_augmented.history['val_accuracy']):.3f}")
plot_history(history_augmented, "CNN augmente")

## Phase 2.3 — Diagnostic comparatif TP1 vs TP2

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_scratch.history['val_loss'], color='red', label='scratch')
plt.plot(history_augmented.history['val_loss'], color='blue', label='augmente')
plt.title("val_loss"); plt.xlabel("Epoch"); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history_scratch.history['val_accuracy'], color='red', label='scratch')
plt.plot(history_augmented.history['val_accuracy'], color='blue', label='augmente')
plt.title("val_accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.tight_layout()
plt.savefig("comparison_tp1_tp2.png", dpi=100)
plt.show()

gap_scratch = (max(history_scratch.history['accuracy'])
               - max(history_scratch.history['val_accuracy']))
gap_aug = (max(history_augmented.history['accuracy'])
           - max(history_augmented.history['val_accuracy']))
print(f"{'':22s} {'scratch':>10s} {'augmente':>10s}")
print(f"{'val_accuracy max':22s} {max(history_scratch.history['val_accuracy']):>10.3f} "
      f"{max(history_augmented.history['val_accuracy']):>10.3f}")
print(f"{'gap train/val':22s} {gap_scratch:>10.3f} {gap_aug:>10.3f}")
print(f"{'temps (s)':22s} {training_time_scratch:>10.0f} {training_time_augmented:>10.0f}")
print(f"{'params':22s} {model_scratch.count_params():>10,} "
      f"{model_augmented.count_params():>10,}")

**Interprétation TP1 vs TP2** : entre TP1 et TP2, la val_loss cesse de remonter
aussi tôt et la val_accuracy monte plus haut : l'augmentation fabrique de la
diversité synthétique (flips, rotations, zooms) que le modèle ne peut pas
mémoriser, et le Dropout l'empêche de s'appuyer sur quelques neurones. Le gap
train/val se réduit nettement (chiffres ci-dessus) — c'est la définition même
d'une régularisation qui fonctionne. La convergence est plus lente : chaque
epoch voit des images différentes, mémoriser ne marche plus, il faut
généraliser. Ce qui reste insuffisant : avec ~2 400 images d'entraînement, un
CNN from scratch ne verra jamais assez de chats et de chiens pour apprendre des
représentations riches — c'est exactement le problème que le transfer learning
résout en TP3 avec un modèle qui a déjà vu 14 millions d'images.

## Phase 3.1 — Chargement MobileNetV2 et freezing

In [ ]:
IMG_SIZE_TL = (160, 160)

train_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    image_size=IMG_SIZE_TL, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=True, seed=42,
)
val_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    image_size=IMG_SIZE_TL, batch_size=BATCH_SIZE,
    label_mode='binary', shuffle=False, seed=42,
)

# preprocess_input normalise dans [-1, 1] : il REMPLACE Rescaling(1./255)
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_tl = train_ds_tl.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
val_ds_tl = val_ds_tl.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,      # obligatoire pour plugger notre tete custom
    weights='imagenet',
)
print(f"Nombre de couches dans la base : {len(base_model.layers)}")
print(f"Parametres de la base : {base_model.count_params():,}")

base_model.trainable = False   # LE piege classique si on l'oublie

inputs = tf.keras.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False)  # BatchNorm reste en mode inference
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model_tl = tf.keras.Model(inputs, outputs)
model_tl.summary()

## Phase 3.2 — Entraînement de la tête uniquement

In [ ]:
# Base gelee -> on peut se permettre un lr eleve pour la tete
model_tl.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                 loss='binary_crossentropy', metrics=['accuracy'])

log_dir_tl = "logs/transfer/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback_tl = tf.keras.callbacks.TensorBoard(log_dir=log_dir_tl)

start = time.time()
history_tl_head = model_tl.fit(
    train_ds_tl, epochs=10, validation_data=val_ds_tl,
    callbacks=[tensorboard_callback_tl], verbose=2)
training_time_head = time.time() - start
print(f"Temps d'entrainement (tete seule) : {training_time_head:.0f}s")
print(f"val_accuracy finale (tete seule) : "
      f"{max(history_tl_head.history['val_accuracy']):.3f}")
plot_history(history_tl_head, "TL tete seule")

## Phase 3.3 — Fine-tuning des dernières couches

In [ ]:
# Premieres couches (bords, textures) : universelles, on les laisse gelees.
# Dernieres couches (formes, concepts) : on les adapte au domaine chats/chiens.
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.8)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False
print(f"Couches degelees : {len(base_model.layers) - fine_tune_at}")
trainable_count = sum(p.numpy().size for p in model_tl.trainable_variables)
print(f"Parametres entrainables : {trainable_count:,}")

# lr x10 plus petit : un grand lr ecraserait les representations ImageNet
# (catastrophic forgetting)
model_tl.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='binary_crossentropy', metrics=['accuracy'])
early_stopping_tl = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True)

start = time.time()
history_tl_finetune = model_tl.fit(
    train_ds_tl, epochs=15, validation_data=val_ds_tl,
    callbacks=[tensorboard_callback_tl, early_stopping_tl], verbose=2)
training_time_finetune = time.time() - start
print(f"Temps d'entrainement (fine-tuning) : {training_time_finetune:.0f}s")
print(f"val_accuracy finale : {max(history_tl_finetune.history['val_accuracy']):.3f}")
plot_history(history_tl_finetune, "TL fine-tuning")

## Phase 3.4 — Tableau de comparaison cross-modèles

In [ ]:
model_scratch.save("model_scratch.keras")
model_augmented.save("model_augmented.keras")
model_tl.save("model_tl.keras")

def model_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

size_scratch = model_size_mb("model_scratch.keras")
size_augmented = model_size_mb("model_augmented.keras")
size_tl = model_size_mb("model_tl.keras")

val_acc_scratch = max(history_scratch.history['val_accuracy'])
val_acc_augmented = max(history_augmented.history['val_accuracy'])
val_acc_tl = max(history_tl_head.history['val_accuracy']
                 + history_tl_finetune.history['val_accuracy'])
params_scratch = model_scratch.count_params()
params_augmented = model_augmented.count_params()
params_tl = model_tl.count_params()
time_tl = training_time_head + training_time_finetune

print("=" * 70)
print("TABLEAU DE COMPARAISON DES TROIS MODELES")
print("=" * 70)
print(f"{'Iteration':<25} {'val_acc':>8} {'Params':>12} {'Temps':>8} {'Taille':>8}")
print("-" * 70)
print(f"{'CNN scratch':<25} {val_acc_scratch:>7.1%} {params_scratch:>12,} "
      f"{training_time_scratch:>7.0f}s {size_scratch:>7.1f}Mo")
print(f"{'CNN augmente + Dropout':<25} {val_acc_augmented:>7.1%} {params_augmented:>12,} "
      f"{training_time_augmented:>7.0f}s {size_augmented:>7.1f}Mo")
print(f"{'MobileNetV2 fine-tuning':<25} {val_acc_tl:>7.1%} {params_tl:>12,} "
      f"{time_tl:>7.0f}s {size_tl:>7.1f}Mo")
print("=" * 70)

plt.figure(figsize=(9, 5))
plt.plot(history_scratch.history['val_accuracy'],
         label='CNN scratch', color='red', linestyle='--')
plt.plot(history_augmented.history['val_accuracy'],
         label='Augmente + Dropout', color='orange')
tl_acc = (history_tl_head.history['val_accuracy']
          + history_tl_finetune.history['val_accuracy'])
plt.plot(range(len(tl_acc)), tl_acc, label='MobileNetV2 fine-tuning', color='green')
plt.xlabel('Epoch'); plt.ylabel('Val Accuracy'); plt.legend()
plt.savefig("comparison_all.png", dpi=100)
plt.show()

**Interprétation du tableau** : MobileNetV2 gagne sur tous les fronts — meilleure
val_accuracy, temps par epoch plus faible (la backprop ne traverse pas la base
gelée pendant la phase tête), et modèle final plus léger que le CNN scratch
malgré 2.2M de paramètres pré-entraînés. La raison n'est pas dans notre code :
elle est dans les représentations. Nos 2 400 images ne serviront jamais à
apprendre « ce qu'est une texture de poil » ; ImageNet (14M d'images) l'a déjà
fait, et on ne paie que l'adaptation de la tête. Honnêteté du tableau : sans la
colonne Params, dire « 95% gagne » ferait croire à de la magie — TP3 part de
2.2M de paramètres pré-entraînés là où TP1 apprend ses 2.4M depuis zéro.

## Phase 3.5 — Export TFLite + exploration (Direction A : quantization INT8)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_tl)
tflite_model = converter.convert()

tflite_path = f"{CLASS_A}_vs_{CLASS_B}_mobilenet.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_tflite = os.path.getsize(tflite_path) / (1024 * 1024)
size_keras = model_size_mb("model_tl.keras")
print(f"Taille Keras : {size_keras:.1f} Mo")
print(f"Taille TFLite : {size_tflite:.1f} Mo")
print(f"Facteur de compression : {size_keras / size_tflite:.1f}x")

In [ ]:
# Inference TFLite sur une image du val set
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

images, labels = next(iter(val_ds_tl))
test_image = images[0:1].numpy()
true_label = int(labels[0][0].numpy())

interpreter.set_tensor(input_details[0]['index'], test_image)
interpreter.invoke()
prediction = interpreter.get_tensor(output_details[0]['index'])[0][0]
print(f"Prediction : {prediction:.3f} (0={CLASS_A}, 1={CLASS_B})")
print(f"Vraie classe : {CLASS_A if true_label == 0 else CLASS_B}")
print(f"Confiance : {max(prediction, 1 - prediction):.1%}")

In [ ]:
# Direction A : quantization INT8 (poids en entiers 8 bits)
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model_tl)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

def representative_dataset():
    # dataset de calibration : estime les plages de valeurs des activations
    for images, _ in val_ds_tl.take(50):
        yield [images.numpy()]

converter_int8.representative_dataset = representative_dataset
tflite_int8 = converter_int8.convert()

tflite_int8_path = f"{CLASS_A}_vs_{CLASS_B}_mobilenet_int8.tflite"
with open(tflite_int8_path, "wb") as f:
    f.write(tflite_int8)

size_int8 = os.path.getsize(tflite_int8_path) / (1024 * 1024)
print(f"Taille TFLite FP32 : {size_tflite:.1f} Mo")
print(f"Taille TFLite INT8 : {size_int8:.1f} Mo")
print(f"Compression INT8 vs FP32 : {size_tflite / size_int8:.1f}x")

# Accuracy du modele quantize sur le val set complet (mesure de l'accuracy drop)
interp8 = tf.lite.Interpreter(model_path=tflite_int8_path)
interp8.allocate_tensors()
in8 = interp8.get_input_details()
out8 = interp8.get_output_details()
correct = total = 0
for images, labels in val_ds_tl:
    for i in range(images.shape[0]):
        interp8.set_tensor(in8[0]['index'], images[i:i+1].numpy())
        interp8.invoke()
        p = interp8.get_tensor(out8[0]['index'])[0][0]
        correct += int((p > 0.5) == bool(labels[i][0].numpy()))
        total += 1
acc_int8 = correct / total
print(f"Accuracy INT8 sur val : {acc_int8:.3f} "
      f"(FP32 fine-tune : {max(history_tl_finetune.history['val_accuracy']):.3f})")

**Limite à documenter (pas un bug)** : sur une image hors distribution (ni chat
ni chien), le modèle binaire force quand même une prédiction, souvent avec une
probabilité élevée — il n'y a aucun mécanisme « rien de connu ». Même
observation qu'avec l'entrée extrême du J2 : un softmax/sigmoid produit toujours
une distribution, jamais un « je ne sais pas ».

**Bilan** : CNN scratch → overfitting assumé ; + augmentation/Dropout → gap
réduit ; MobileNetV2 → meilleure accuracy pour un modèle plus léger, exportable
en `.tflite` mobile. La progression du tableau 3.4 raconte toute la journée.